# Notebook 3: Sentiment Analysis

## Purpose
Classify each KFC Reddit comment as **Positive**, **Negative**, or **Neutral** using multiple models, then select the best one to label the full dataset.

## Pipeline
1. Load 10% manually labelled sample (your gold standard)
2. Train/test split (60/40)
3. Test four models: RoBERTa, BERTweet, Logistic Regression, Naive Bayes
4. Apply SMOTE to balance classes for traditional models
5. Compare accuracy across all models
6. Use the best model (BERTweet) to label the ENTIRE dataset
7. Save the fully labelled dataset

## Input
- `kfc_preprocessed_sentiment.csv` (from Notebook 2)
- `kfc_10_percent_labelled.csv` (YOU create this — see Step 2)

## Output
- `kfc_sentiment_labelled_full.csv`
- `sentiment_distribution.png`

## Step 0 — Install & Import Libraries

In [ ]:
# !pip install pandas numpy scikit-learn imbalanced-learn transformers torch matplotlib seaborn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from transformers import pipeline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded.")

## Step 1 — Configuration

In [ ]:
PREPROCESSED_FILE = "kfc_preprocessed_sentiment.csv"
LABELLED_SAMPLE   = "kfc_10_percent_labelled.csv"
OUTPUT_FILE       = "kfc_sentiment_labelled_full.csv"
CHART_FILE        = "sentiment_distribution.png" 

## Step 2 — Create Your Manually Labelled Sample

⚠️ **You must do this step BEFORE running the rest of this notebook.**

Run the cell below to generate a 10% random sample. Then:
1. Open the resulting CSV in Excel or Google Sheets
2. Read each comment and type a label in the `Manual_Label` column:
   - **Positive** — praise, satisfaction, excitement
   - **Negative** — complaints, frustration, criticism
   - **Neutral** — factual, questions, neither positive nor negative
3. Save the file as `kfc_10_percent_labelled.csv`

In [ ]:
# --- RUN THIS ONCE to generate the sample file ---
# Then open it, label it, save it, and continue.

df_temp = pd.read_csv(PREPROCESSED_FILE)
sample = df_temp.sample(frac=0.10, random_state=42)
sample["Manual_Label"] = ""
sample.to_csv(LABELLED_SAMPLE, index=False)
print(f"Sample saved: {LABELLED_SAMPLE}")
print(f"Comments to label: {len(sample)}")
print("\nOpen this file, add labels in the Manual_Label column, then re-run the notebook.")

## Step 3 — Load Datasets

In [ ]:
df_full = pd.read_csv(PREPROCESSED_FILE)
print(f"Full dataset: {len(df_full)} comments")

df_labelled = pd.read_csv(LABELLED_SAMPLE)
print(f"Labelled sample: {len(df_labelled)} comments")
print(f"\nLabel distribution:")
print(df_labelled["Manual_Label"].value_counts())

## Step 4 — Train/Test Split (60/40)

A 60/40 split ensures a large enough test set for reliable evaluation, since the labelled sample is small.

In [ ]:
X = df_labelled["Tokenised_Comment"]
y = df_labelled["Manual_Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} | Test set: {len(X_test)}")

## Step 5 — Traditional Models (Logistic Regression + Naive Bayes)

These use **Bag-of-Words (BoW)** vectorisation.  
**SMOTE** is applied to the training set to balance the class distribution.

In [ ]:
# --- Vectorise ---
vectorizer = CountVectorizer(max_features=5000)
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow  = vectorizer.transform(X_test)

# --- Apply SMOTE to training data ---
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_bow, y_train)
print("After SMOTE — train samples per class:")
print(pd.Series(y_train_balanced).value_counts())

In [ ]:
# --- Logistic Regression ---
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_balanced, y_train_balanced)
lr_preds = lr_model.predict(X_test_bow)
lr_accuracy = accuracy_score(y_test, lr_preds)

print(f"\nLogistic Regression Accuracy: {lr_accuracy:.3f}")
print(classification_report(y_test, lr_preds))

In [ ]:
# --- Naive Bayes ---
print("Training Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train_balanced, y_train_balanced)
nb_preds = nb_model.predict(X_test_bow)
nb_accuracy = accuracy_score(y_test, nb_preds)

print(f"\nNaive Bayes Accuracy: {nb_accuracy:.3f}")
print(classification_report(y_test, nb_preds))

## Step 6 — Transformer Models (RoBERTa + BERTweet)

- **RoBERTa** — general-purpose (trained on books/news)
- **BERTweet** — trained on 850M tweets, handles slang, abbreviations, sarcasm

In [ ]:
def map_label(label_str):
    """Normalise transformer output labels to Positive/Negative/Neutral."""
    l = label_str.lower()
    if "neg" in l or "label_0" in l: return "Negative"
    elif "pos" in l or "label_2" in l: return "Positive"
    else: return "Neutral"

print("Label mapper defined.")

In [ ]:
# --- RoBERTa ---
print("Loading RoBERTa...")
roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    max_length=512, truncation=True,
)

print("Running RoBERTa on test set...")
roberta_preds = [map_label(roberta(str(t)[:512])[0]["label"]) for t in X_test]
roberta_accuracy = accuracy_score(y_test, roberta_preds)

print(f"\nRoBERTa Accuracy: {roberta_accuracy:.3f}")
print(classification_report(y_test, roberta_preds))

In [ ]:
# --- BERTweet ---
print("Loading BERTweet...")
bertweet = pipeline(
    "sentiment-analysis",
    model="finiteautomata/bertweet-base-sentiment-analysis",
    tokenizer="finiteautomata/bertweet-base-sentiment-analysis",
    max_length=128, truncation=True,
)

print("Running BERTweet on test set...")
bertweet_preds = [map_label(bertweet(str(t)[:128])[0]["label"]) for t in X_test]
bertweet_accuracy = accuracy_score(y_test, bertweet_preds)

print(f"\nBERTweet Accuracy: {bertweet_accuracy:.3f}")
print(classification_report(y_test, bertweet_preds))

## Step 7 — Model Comparison

In [ ]:
results = {
    "Logistic Regression": lr_accuracy,
    "Naive Bayes": nb_accuracy,
    "RoBERTa": roberta_accuracy,
    "BERTweet": bertweet_accuracy,
}

print("MODEL COMPARISON")
print("=" * 40)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    marker = " <-- BEST" if acc == max(results.values()) else ""
    print(f"  {name:25s}  {acc:.3f}{marker}")

## Step 8 — Label Full Dataset with BERTweet

BERTweet was the best performer in the original McDonald's study (76% accuracy).  
We use it to label all comments in the full dataset.

In [ ]:
print("Labelling full dataset with BERTweet...")
all_labels = []
total = len(df_full)

for i, text in enumerate(df_full["Tokenised_Comment"]):
    result = bertweet(str(text)[:128])[0]
    all_labels.append(map_label(result["label"]))
    if (i + 1) % 500 == 0:
        print(f"  ... {i+1}/{total}")

df_full["Sentiment"] = all_labels
print(f"\nDone! Labelled {total} comments.")

## Step 9 — Visualise Sentiment Distribution

In [ ]:
counts = df_full["Sentiment"].value_counts()
colors = {"Neutral": "#888888", "Negative": "#E74C3C", "Positive": "#2ECC71"}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values,
              color=[colors.get(c, "#3498DB") for c in counts.index])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(val), ha="center", fontsize=12, fontweight="bold")

ax.set_title("Distribution of Sentiment Labels (BERTweet) — KFC", fontsize=14)
ax.set_xlabel("Sentiment"); ax.set_ylabel("Number of Comments")
plt.tight_layout()
plt.savefig(CHART_FILE, dpi=150)
plt.show()

## Step 10 — Save Fully Labelled Dataset

In [ ]:
df_full.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Saved to: {OUTPUT_FILE}")
print(f"Total: {len(df_full)} comments")
print(f"\nSentiment breakdown:")
print(df_full["Sentiment"].value_counts())